<div style="font-family: system-ui, -apple-system, sans-serif; text-align: center; padding: 48px 24px 24px;">
    <div style="display: inline-block; background: #be0f05; color: white;
                padding: 12px 20px; border-radius: 12px; margin-bottom: 20px;">
        <span style="font-size: 36px; font-weight: 800; letter-spacing: -0.5px;">TIP4PATLIBS &ndash; Patent Landscape Reports</span>
    </div>
    <div style="font-size: 16px; color: #475569; margin-bottom: 8px; line-height: 1.6;">
        Turn a patent dataset into a <strong>publishable landscape report</strong> &mdash; maps, rankings,
        technology clusters and an interactive explorer.
    </div>
    <div style="font-size: 13px; color: #94a3b8; margin-bottom: 32px;">
        EPO Academy Training Material &nbsp;&middot;&nbsp; <span style="color: #be0f05; font-weight: 600;">created by Riccardo Priore</span>
        &nbsp;&middot;&nbsp; Centro PATLIB, AREA Science Park
    </div>
    <div style="background: #f8fafc; border-radius: 12px; padding: 24px 28px; max-width: 660px;
                margin: 0 auto; border: 1px solid #e2e8f0; text-align: left;">
        <div style="font-size: 14px; color: #334155; line-height: 1.9;">
            <strong>What this notebook builds</strong>
            <br/>Step&nbsp;1 &nbsp;&middot;&nbsp; Setup &mdash; PATSTAT connection, base dataset, chart-decoding helpers
            <br/>Step&nbsp;2 &nbsp;&middot;&nbsp; <strong>Triadic patent families</strong> &mdash; world map + searchable applicant ranking
            <br/>Step&nbsp;3 &nbsp;&middot;&nbsp; <strong>International filing authorities</strong> &mdash; WO/EP/EA/AP/OA breakdown &amp; trend
            <br/>Step&nbsp;4 &nbsp;&middot;&nbsp; Extract the report's <strong>actual cluster assignments</strong>
            <br/>Step&nbsp;5 &nbsp;&middot;&nbsp; <strong>Interactive Cluster Explorer</strong> &mdash; clickable clusters &rarr; patent lists
            <br/>Step&nbsp;6 &nbsp;&middot;&nbsp; <strong>IPC co-occurrence over time</strong> &mdash; relationship-change heatmap
        </div>
    </div>
    <div style="background: #fdf2f2; border-radius: 10px; padding: 16px 24px; max-width: 660px;
                margin: 28px auto 0; border: 1px solid #fecaca;">
        <div style="font-size: 14px; color: #404955; font-weight: 600;">&#9654; &nbsp;The outputs below are already computed &mdash; read it like a report.</div>
        <div style="font-size: 12px; color: #64748b; margin-top: 6px; line-height: 1.6;">
            To re-run it yourself you need <strong>EPO&nbsp;TIP</strong> (it queries PATSTAT&nbsp;PROD via
            <code>epo.tipdata</code>) and the <code>output_*</code> folders shipped alongside.
            Nothing here re-runs the original keyword/IPC search &mdash; every step reuses the
            existing <strong>3,974-family</strong> antibiotic-resistance dataset.
        </div>
    </div>
    <div style="margin-top: 20px; font-size: 12px; color: #cbd5e1;">
        Part of EPO TIP Working Group Sessions, 2026. &nbsp;Data: EPO PATSTAT Global.
    </div>
</div>

# Antibiotic Resistance — Modernized Report: Additional Analyses

Produces the four analyses added on top of `Antibiotic_Report_FINAL.html` when building
the assembled report (`report/antibiotic_resistance_report.html`, built in Step 7) (2026-07-16/17), which previously
existed only as ad-hoc queries with no notebook behind them:

1. **Triadic Patent Families** — world map + searchable applicant ranking table
2. **International/Regional Filing Authority Breakdown** — WO/EP/EA/AP/OA counts + trend
3. **ANALYSIS 10 cluster family extraction** — the exact family IDs behind each of the 7
   t-SNE clusters, read directly out of the existing report's own chart data (not a new
   clustering run) so any downstream use matches the Cluster Distribution table exactly
4. **Interactive Cluster Explorer data** — title/year/applicant per family, for the
   clickable cluster → patent-list widget
5. **IPC/Technology Co-occurrence Temporal Evolution** — relationship-change heatmap
   across the same 3 periods already established in Analysis 9

**Does not** re-run the original keyword/IPC search — everything here reuses the existing
3,974-family dataset (`1_dataset_and_search_strategy_output/dataset.xlsx`)
or extracts data already embedded in `Antibiotic_Report_FINAL.html`.

## Step 1: Setup — PATSTAT connection, base dataset, shared bdata-decode utility

### Connect and load the base dataset

Opens the PATSTAT ORM session on TIP and loads the **3,974 antibiotic-resistance patent
families** that were defined once in the original dataset notebook. Every later step
filters on this same family list, so all figures in the report stay mutually consistent.

In [1]:
import pandas as pd
import numpy as np
import json
import re
import base64
import struct
import os
import pycountry
import plotly.express as px
import plotly.graph_objects as go

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN, TLS202_APPLN_TITLE, TLS206_PERSON, TLS207_PERS_APPLN,
    TLS209_APPLN_IPC, TLS211_PAT_PUBLN
)
from sqlalchemy import and_, or_, func

patstat = PatstatClient(env='PROD')
db = patstat.orm()

dataset_df = pd.read_excel("1_dataset_and_search_strategy_output/dataset.xlsx")
base_family_ids = dataset_df['docdb_family_id'].unique().tolist()
print(f"Base dataset: {len(base_family_ids):,} families")

Base dataset: 4,172 families


### Helper: decode Plotly's compact chart data

Plotly stores numeric arrays in a compact binary form (`{"dtype": …, "bdata": …}`, plus a
`shape` key for 2-D data such as heatmap matrices). Step&nbsp;4 has to *read back* the chart
data that is already embedded in the published report, so these helpers decode that format
and safely walk the JavaScript `Plotly.newPlot(...)` call to find it.

In [2]:
# Shared utility: Plotly compact-encodes numeric arrays as {"dtype":..,"bdata":..}
# (optionally with a "shape" key for 2D data like heatmap z-matrices). Needed both to
# read existing embedded charts (Step 3) and to clean any chart we export ourselves.
def reshape(flat, shape):
    if len(shape) == 1:
        return flat[:shape[0]]
    sub_size = 1
    for s in shape[1:]:
        sub_size *= s
    return [reshape(flat[i*sub_size:(i+1)*sub_size], shape[1:]) for i in range(shape[0])]

def decode_bdata_recursive(obj):
    if isinstance(obj, dict):
        if {"dtype", "bdata"} <= set(obj.keys()) and isinstance(obj.get("bdata"), str):
            fmt_map = {"i1":"b","u1":"B","i2":"h","u2":"H","i4":"i","u4":"I","f4":"f","f8":"d"}
            dtype = obj["dtype"]
            if dtype in fmt_map:
                raw = base64.b64decode(obj["bdata"])
                fmt = fmt_map[dtype]
                n = len(raw) // struct.calcsize(fmt)
                flat = list(struct.unpack(f"<{n}{fmt}", raw))
                if "shape" in obj:
                    return reshape(flat, [int(x) for x in obj["shape"].split(",")])
                return flat
        return {k: decode_bdata_recursive(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [decode_bdata_recursive(v) for v in obj]
    return obj

def find_matching_paren(s, open_idx):
    depth = 0; i = open_idx; in_str = False; str_char = None; escape = False
    while i < len(s):
        c = s[i]
        if in_str:
            if escape: escape = False
            elif c == '\\': escape = True
            elif c == str_char: in_str = False
        else:
            if c in ('"', "'"): in_str = True; str_char = c
            elif c == '(': depth += 1
            elif c == ')':
                depth -= 1
                if depth == 0: return i
        i += 1
    return -1

def split_top_level_args(inner):
    args = []; depth = 0; cur = []; in_str = False; str_char = None; escape = False
    for c in inner:
        if in_str:
            cur.append(c)
            if escape: escape = False
            elif c == '\\': escape = True
            elif c == str_char: in_str = False
            continue
        if c in ('"', "'"): in_str = True; str_char = c; cur.append(c); continue
        if c in '[{(': depth += 1
        if c in ']})': depth -= 1
        if c == ',' and depth == 0:
            args.append(''.join(cur)); cur = []
        else:
            cur.append(c)
    args.append(''.join(cur))
    return args
print("Shared bdata-decode utilities ready")

Shared bdata-decode utilities ready


## Step 2: Triadic Patent Families

Families with a granted or B-kind US publication, as a triadic-strength proxy — same
convention already used in `Genome-Triadic families.ipynb`. Produces a world map (log
color scale — a linear scale is unreadable when one country dominates) and a searchable
DataTables ranking of applicants by country of residence.

### Find the triadic families and their applicants

A family counts as **triadic-strength** here if it has a granted or `B`-kind **US**
publication &mdash; the same proxy already used in the Genome notebooks. Two queries follow:
first the qualifying family IDs, then the applicants behind them (`applt_seq_nr != 0`
excludes inventors) together with their country of residence.

In [3]:
os.makedirs("3_additional_analyses_and_report_output", exist_ok=True)

final_query = (
    db.query(TLS201_APPLN.docdb_family_id)
    .join(TLS211_PAT_PUBLN, TLS201_APPLN.appln_id == TLS211_PAT_PUBLN.appln_id)
    .filter(
        TLS201_APPLN.docdb_family_id.in_(base_family_ids),
        and_(
            TLS211_PAT_PUBLN.publn_auth == 'US',
            or_(TLS201_APPLN.granted == 'Y', TLS211_PAT_PUBLN.publn_kind.startswith('B'))
        )
    )
    .distinct()
)
triadic_family_ids = [r.docdb_family_id for r in final_query.all()]
print(f"Triadic-proxy families: {len(triadic_family_ids):,}")

applicant_query = (
    db.query(TLS201_APPLN.docdb_family_id, TLS206_PERSON.psn_name, TLS206_PERSON.person_ctry_code)
    .join(TLS207_PERS_APPLN, TLS201_APPLN.appln_id == TLS207_PERS_APPLN.appln_id)
    .join(TLS206_PERSON, TLS207_PERS_APPLN.person_id == TLS206_PERSON.person_id)
    .filter(TLS201_APPLN.docdb_family_id.in_(triadic_family_ids), TLS207_PERS_APPLN.applt_seq_nr != 0)
    .distinct()
    .order_by(TLS201_APPLN.docdb_family_id)
)
applicant_results = applicant_query.all()
triadic_df = pd.DataFrame({
    "docdb_family_id": [r.docdb_family_id for r in applicant_results],
    "Applicant": [r.psn_name for r in applicant_results],
    "Country Code": [r.person_ctry_code for r in applicant_results],
})
triadic_df.to_excel("3_additional_analyses_and_report_output/triadic_families_applicants.xlsx", index=False)
print(f"Applicant rows: {len(triadic_df):,}")

Triadic-proxy families: 574
Applicant rows: 2,636


Applicant rows: 2,634


### World map of applicant countries

A **logarithmic** colour scale is deliberate: one country dominates the field so strongly
that a linear scale would render every other country invisible. Country codes are converted
from ISO&nbsp;alpha-2 to alpha-3 for the choropleth, and unmappable codes are dropped.

In [4]:
# World map (log color scale)
df = triadic_df.copy()
df["Country Code"] = df["Country Code"].astype(str).str.strip()
df = df[df["Country Code"].notnull() & (df["Country Code"] != "") & (df["Country Code"] != "nan")]

def to_3letter(code):
    try:
        return pycountry.countries.get(alpha_2=code).alpha_3
    except AttributeError:
        return None

df["Country Code 3"] = df["Country Code"].apply(to_3letter)
df = df[df["Country Code 3"].notnull()]
country_counts = df.groupby("Country Code 3").size().reset_index(name="Applicant Count")
country_counts = country_counts.merge(df[["Country Code", "Country Code 3"]].drop_duplicates(), on="Country Code 3")
country_counts["Log Applicant Count"] = np.log10(country_counts["Applicant Count"])

fig = px.choropleth(
    country_counts, locations="Country Code 3", color="Log Applicant Count", hover_name="Country Code",
    hover_data={"Applicant Count": True, "Log Applicant Count": False, "Country Code 3": False},
    color_continuous_scale=px.colors.sequential.Plasma,
    title="Antibiotic Resistance -- Triadic Patent Families: Country of Residence x Applicants",
)
fig.update_layout(width=1100, height=650, coloraxis_colorbar=dict(title="Applicants", tickvals=[0,1,2,3], ticktext=["1","10","100","1,000"]))
fig.write_html("3_additional_analyses_and_report_output/triadic_families_map.html")
triadic_map_fig = fig   # kept for the report assembly in Step 7 (`fig` is reused below)
print(country_counts.sort_values("Applicant Count", ascending=False).head(10))

   Country Code 3  Applicant Count Country Code  Log Applicant Count
39            USA              949           US             2.977266
24            JPN              205           JP             2.311754
8             CHN              123           CN             2.089905
26            KOR              118           KR             2.071882
15            FRA              112           FR             2.049218
18            IND              109           IN             2.037426
10            DEU               64           DE             1.806180
16            GBR               64           GB             1.806180
5             CAN               52           CA             1.716003
6             CHE               20           CH             1.301030


### Searchable applicant ranking

Ranks applicants by their number of **distinct** triadic families and writes an interactive
DataTables page &mdash; sortable and full-text searchable, so a PATLIB colleague can look up a
specific company without touching the data.

In [5]:
# Searchable applicant ranking table (DataTables, matching DEEPTECH-series convention)
ranking = (
    df.groupby(["Applicant", "Country Code"])["docdb_family_id"]
    .nunique().reset_index(name="Triadic Families")
    .sort_values("Triadic Families", ascending=False).reset_index(drop=True)
)
ranking.insert(0, "Rank", ranking.index + 1)
ranking.to_excel("3_additional_analyses_and_report_output/triadic_applicant_ranking.xlsx", index=False)
print(f"Distinct applicant/country combinations: {len(ranking):,}")

rows_html = "".join(
    f"<tr><td>{row.Rank}</td><td>{row.Applicant}</td><td>{row['Country Code']}</td><td>{row['Triadic Families']}</td></tr>"
    for _, row in ranking.iterrows()
)
table_html = f"""<!DOCTYPE html>
<html><head><meta charset="UTF-8">
<title>Antibiotic Resistance - Triadic Families Applicant Ranking</title>
<link rel="stylesheet" href="https://cdn.datatables.net/1.13.6/css/jquery.dataTables.min.css">
<script src="https://code.jquery.com/jquery-3.7.0.min.js"></script>
<script src="https://cdn.datatables.net/1.13.6/js/jquery.dataTables.min.js"></script>
<style>body{{font-family:'Segoe UI',sans-serif;margin:20px;background:#fff;}} .container{{max-width:100%;}} table.display{{width:100%;}} h2{{color:#4527A0;}}</style>
</head><body><div class="container">
<h2>Triadic-Strength Applicant Ranking, by Country of Residence</h2>
<p>Ranked by number of distinct triadic-proxy patent families ({len(triadic_family_ids):,} families total, {len(ranking):,} distinct applicant/country combinations).</p>
<table class="display"><thead><tr><th>Rank</th><th>Applicant</th><th>Country</th><th>Triadic Families</th></tr></thead><tbody>{rows_html}</tbody></table>
</div>
<script>$(document).ready(function(){{ $('table.display').each(function(){{ $(this).DataTable({{pageLength:25, order:[[3,'desc']]}}); }}); }});</script>
</body></html>"""

with open("3_additional_analyses_and_report_output/triadic_applicant_ranking_table.html", "w", encoding="utf-8") as f:
    f.write(table_html)
print("Saved applicant ranking table")

Distinct applicant/country combinations: 1,539
Saved applicant ranking table


## Step 3: International/Regional Filing Authority Breakdown

Analysis 3 in the original report only distinguishes "international (PCT)" vs. "national"
— it doesn't show which *specific* treaty/regional office received filings. This queries
every publication authority for the base family list and isolates the treaty/regional ones.

### Query every publication authority

The original report only separated *international (PCT)* from *national* filings. This
retrieves the actual publication authority for each application in the dataset, which makes
the regional treaty offices visible.

In [6]:
result = (
    db.query(TLS201_APPLN.docdb_family_id, TLS201_APPLN.appln_id, TLS211_PAT_PUBLN.publn_auth, TLS201_APPLN.appln_filing_year)
    .join(TLS211_PAT_PUBLN, TLS201_APPLN.appln_id == TLS211_PAT_PUBLN.appln_id)
    .filter(TLS201_APPLN.docdb_family_id.in_(base_family_ids))
    .distinct()
    .all()
)
auth_df = pd.DataFrame(result, columns=["docdb_family_id", "appln_id", "publn_auth", "filing_year"])
auth_df.to_excel("3_additional_analyses_and_report_output/authority_breakdown.xlsx", index=False)
print(f"Publication-authority rows: {len(auth_df):,}")
print(auth_df['publn_auth'].value_counts().head(10))

Publication-authority rows: 11,020
publn_auth
CN    2821
US    1615
WO    1252
EP     830
KR     670
JP     650
AU     409
CA     403
RU     182
BR     182
Name: count, dtype: int64


### Isolate the treaty and regional offices

Keeps the five supranational authorities &mdash; **WO** (PCT/WIPO), **EP** (European Patent
Office), **EA** (Eurasian), **AP** (ARIPO, Africa) and **OA** (OAPI, Africa) &mdash; and produces
a totals bar chart plus a WO-vs-EP trend line over 2000&ndash;2023.

In [7]:
intl_authorities = ["WO", "EP", "EA", "AP", "OA"]
intl_df = auth_df[auth_df["publn_auth"].isin(intl_authorities)]
labels = {"WO": "WO — PCT (WIPO)", "EP": "EP — European Patent Office", "EA": "EA — Eurasian Patent Org.", "AP": "AP — ARIPO (Africa)", "OA": "OA — OAPI (Africa)"}
totals = intl_df["publn_auth"].value_counts().reindex(intl_authorities).fillna(0).astype(int)
print(totals)

fig1 = px.bar(
    x=[labels[a] for a in totals.index], y=totals.values,
    labels={"x": "International / Regional Authority", "y": "Number of Filings"},
    title="Antibiotic Resistance Patents -- Filings by International/Regional Authority (2000-2023)",
    text=totals.values,
)
fig1.update_traces(marker_color="#4527A0", textposition="outside")
fig1.update_layout(width=900, height=500)
fig1.write_html("3_additional_analyses_and_report_output/intl_authority_totals.html")

yearly = intl_df[intl_df["publn_auth"].isin(["WO", "EP"])].groupby(["filing_year", "publn_auth"]).size().reset_index(name="count")
yearly = yearly[(yearly["filing_year"] >= 2000) & (yearly["filing_year"] <= 2023)]
fig2 = px.line(
    yearly, x="filing_year", y="count", color="publn_auth", markers=True,
    labels={"filing_year": "Filing Year", "count": "Number of Filings", "publn_auth": "Authority"},
    title="WO (PCT) vs EP (European Patent Office) Filings Over Time",
)
fig2.update_layout(width=1000, height=500)
fig2.write_html("3_additional_analyses_and_report_output/intl_authority_trend.html")
print("Saved both authority charts")

publn_auth
WO    1252
EP     830
EA      41
AP       8
OA       2
Name: count, dtype: int64
Saved both authority charts


## Step 4: Extract ANALYSIS 10's actual cluster family assignments

**Important**: `Antibiotic_Resistance_tSNE_Analysis_NEW.ipynb`'s own `Cluster_Data.xlsx`
output is a *different, separate* t-SNE run (3,840 families) from the clustering ANALYSIS
10 actually shows in `Antibiotic_Report_FINAL.html` (3,585 families: 470/681/396/819/353/332/534).
Using that notebook's output for a cluster explorer silently mismatches the diagram — this
was a real bug caught 2026-07-17. The fix: read the family IDs directly out of ANALYSIS 10's
own embedded chart (`text` field per trace, one trace per cluster) instead of re-running or
reusing a different clustering.

### Read the cluster assignments out of the published chart

Rather than re-running the clustering (which would produce different groups), this extracts
each cluster's family IDs **directly from the chart embedded in the original report**, so the
Cluster Explorer matches the published diagram exactly. The result is cached as
`analysis10_cluster_families.json`.

> **Dependency:** this step reads `Antibiotic_Report_FINAL.html` (~50 MB), the original
> published report. It ships in this folder, so the step re-runs as-is. The cached JSON is
> included too, so Steps 5&ndash;6 work even without it.

In [8]:
ORIGINAL_REPORT = "0_inputs/Antibiotic_Report_FINAL.html"
CLUSTER_CHART_DIV_ID = "6be09516-7c8f-4c74-85dd-5607002b4569"  # the 'Interactive Technology Clustering' chart

with open(ORIGINAL_REPORT, encoding="utf-8", errors="ignore") as f:
    original_html = f.read()

idx = original_html.find(CLUSTER_CHART_DIV_ID)
call_m = re.search(r'Plotly\.newPlot\(\s*"' + re.escape(CLUSTER_CHART_DIV_ID) + '"', original_html[idx:])
call_start = idx + call_m.start()
paren_open = original_html.find('(', call_start)
paren_close = find_matching_paren(original_html, paren_open)
call_text = original_html[call_start:paren_close+1]
args = split_top_level_args(call_text[call_text.find('(')+1:-1])
chart_data = decode_bdata_recursive(json.loads(args[1]))

cluster_families = {}
for trace in chart_data:
    cluster_families[trace['name']] = [int(round(x)) for x in trace['text']]
    print(f"{trace['name']}: {len(cluster_families[trace['name']]):,} families")
print("TOTAL:", sum(len(v) for v in cluster_families.values()))

os.makedirs("3_additional_analyses_and_report_output", exist_ok=True)
with open("3_additional_analyses_and_report_output/cluster_families.json", "w") as f:
    json.dump(cluster_families, f)

Cluster 0: 470 families
Cluster 1: 681 families
Cluster 2: 396 families
Cluster 3: 819 families
Cluster 4: 353 families
Cluster 5: 332 families
Cluster 6: 534 families
TOTAL: 3585


## Step 5: Cluster Explorer data — title/year/applicant per family, + the interactive widget

### Fetch title, year and applicant for every clustered family

Three queries assemble what the explorer needs to show per patent family: a representative
publication number (for the Espacenet link), the title and earliest filing year, and the
applicant name.

In [9]:
all_cluster_family_ids = sorted(set(fid for ids in cluster_families.values() for fid in ids))
print(f"Total unique families across all clusters: {len(all_cluster_family_ids):,}")

pub_result = (
    db.query(TLS201_APPLN.docdb_family_id, TLS211_PAT_PUBLN.publn_auth, TLS211_PAT_PUBLN.publn_nr, TLS211_PAT_PUBLN.publn_kind)
    .join(TLS211_PAT_PUBLN, TLS201_APPLN.appln_id == TLS211_PAT_PUBLN.appln_id)
    .filter(TLS201_APPLN.docdb_family_id.in_(all_cluster_family_ids))
    .distinct().all()
)
pub_df = pd.DataFrame(pub_result, columns=["Family_ID", "Authority", "Number", "Kind"])
pub_df["Publication_Number"] = pub_df["Authority"] + pub_df["Number"] + pub_df["Kind"]
pub_df = pub_df.drop_duplicates(subset=["Family_ID"], keep="first")

title_result = (
    db.query(TLS201_APPLN.docdb_family_id, TLS201_APPLN.earliest_filing_year, TLS202_APPLN_TITLE.appln_title)
    .join(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
    .filter(TLS201_APPLN.docdb_family_id.in_(all_cluster_family_ids))
    .distinct().all()
)
title_df = pd.DataFrame(title_result, columns=["Family_ID", "Year", "Title"]).drop_duplicates(subset=["Family_ID"], keep="first")

appl_result = (
    db.query(TLS201_APPLN.docdb_family_id, TLS206_PERSON.psn_name, TLS207_PERS_APPLN.applt_seq_nr)
    .join(TLS207_PERS_APPLN, TLS201_APPLN.appln_id == TLS207_PERS_APPLN.appln_id)
    .join(TLS206_PERSON, TLS207_PERS_APPLN.person_id == TLS206_PERSON.person_id)
    .filter(TLS201_APPLN.docdb_family_id.in_(all_cluster_family_ids), TLS207_PERS_APPLN.applt_seq_nr != 0)
    .distinct().all()
)
appl_df = pd.DataFrame(appl_result, columns=["Family_ID", "Applicant", "seq"]).sort_values("seq").drop_duplicates(subset=["Family_ID"], keep="first")
print(f"Publication rows: {len(pub_df):,}, Title/year rows: {len(title_df):,}, Applicant rows: {len(appl_df):,}")

Total unique families across all clusters: 3,585
Publication rows: 3,585, Title/year rows: 3,585, Applicant rows: 3,566


Publication rows: 3,585, Title/year rows: 3,585, Applicant rows: 3,566


In [10]:
pub_map = pub_df.set_index("Family_ID")[["Publication_Number", "Authority"]].to_dict("index")
title_map = title_df.set_index("Family_ID")[["Year", "Title"]].to_dict("index")
appl_map = appl_df.set_index("Family_ID")["Applicant"].to_dict()

cluster_data = {}
for cluster_name, family_ids in cluster_families.items():
    cluster_num = cluster_name.replace("Cluster ", "")
    items = []
    for fid in family_ids:
        pub = pub_map.get(fid, {})
        ttl = title_map.get(fid, {})
        items.append({
            "docdb_family_id": fid,
            "publication_number": pub.get("Publication_Number", "N/A"),
            "authority": pub.get("Authority", "N/A"),
            "year": int(ttl["Year"]) if fid in title_map and title_map[fid]["Year"] else None,
            "title": ttl.get("Title", ""),
            "applicant": appl_map.get(fid, ""),
        })
    cluster_data[cluster_num] = items

with open("3_additional_analyses_and_report_output/cluster_data.json", "w", encoding="utf-8") as f:
    json.dump(cluster_data, f, ensure_ascii=False)
for cid in sorted(cluster_data.keys(), key=int):
    print(f"Cluster {cid}: {len(cluster_data[cid])} patents")

Cluster 0: 470 patents
Cluster 1: 681 patents
Cluster 2: 396 patents
Cluster 3: 819 patents
Cluster 4: 353 patents
Cluster 5: 332 patents
Cluster 6: 534 patents


In [11]:
# Build the interactive widget: clickable cluster cards -> searchable DataTable, Espacenet-linked
cards_html = []
for cid in sorted(cluster_data.keys(), key=int):
    items = cluster_data[cid]
    auth_counts = pd.Series([i["authority"] for i in items]).value_counts()
    top2 = auth_counts.head(2)
    badges = "".join(
        f'<span class="badge {"bg-primary" if auth=="EP" else "bg-info" if auth=="WO" else "bg-success" if auth=="US" else "bg-warning" if auth=="CN" else "bg-secondary"}" style="font-size:0.8em">{auth}: {count}</span> '
        for auth, count in top2.items()
    )
    cards_html.append(f'<div class="col-md-2 col-sm-4 mb-3"><div class="card cluster-card h-100" onclick="selectCluster(this,{cid})"><div class="card-body text-center"><h5 class="card-title">Cluster {cid}</h5><p class="card-text"><strong>{len(items)}</strong> patents<br>{badges}</p></div></div></div>')

cluster_data_json = json.dumps(cluster_data, ensure_ascii=False)

widget_html = f"""<!DOCTYPE html>
<html><head><meta charset="UTF-8"><title>Antibiotic Resistance - Interactive Cluster Explorer</title>
<link href="https://cdn.jsdelivr.net/npm/bootstrap@5.1.3/dist/css/bootstrap.min.css" rel="stylesheet">
<link rel="stylesheet" href="https://cdn.datatables.net/1.13.6/css/jquery.dataTables.min.css">
<script src="https://code.jquery.com/jquery-3.7.0.min.js"></script>
<script src="https://cdn.datatables.net/1.13.6/js/jquery.dataTables.min.js"></script>
<style>
body {{ font-family:'Segoe UI',sans-serif; margin:20px; }}
.cluster-card {{ cursor:pointer; transition: all .15s; }}
.cluster-card:hover {{ box-shadow:0 4px 10px rgba(0,0,0,.15); }}
.cluster-selected {{ border:2px solid #4527A0 !important; background:#F1EEFC; }}
.cluster-summary {{ background:#F1EEFC; border-left:5px solid #4527A0; padding:14px 20px; border-radius:0 6px 6px 0; }}
.pub-link {{ color:#4527A0; font-weight:600; text-decoration:none; }}
.pub-link:hover {{ text-decoration:underline; }}
#clusterData {{ display:none; margin-top:20px; }}
</style></head><body><div class="container-fluid">
<h4>Select a Cluster to Explore its Patent Documents</h4>
<p class="text-muted">Click any cluster below to list its patent families, with direct links to Espacenet.</p>
<div class="row">{''.join(cards_html)}</div>
<div id="clusterData">
<div class="row mb-3"><div class="col-12"><div class="cluster-summary"><h4 id="clusterTitle">Cluster</h4><p id="clusterDesc">Select a cluster to view patents</p></div></div></div>
<div class="row"><div class="col-12"><div class="card"><div class="card-header d-flex justify-content-between align-items-center"><h5 class="mb-0">Patent Publications</h5><button class="btn btn-secondary btn-sm" onclick="exportCSV()">Export CSV</button></div>
<div class="card-body"><table id="patentTable" class="table table-striped table-hover"><thead class="table-dark"><tr><th>Family ID</th><th>Publication</th><th>Authority</th><th>Year</th><th>Title</th><th>Applicant</th></tr></thead><tbody id="patentTbody"></tbody></table></div>
</div></div></div></div></div>
<script>
const clusterData = {cluster_data_json};
let cur=null, dt=null;
function selectCluster(el,id){{
  document.querySelectorAll(".cluster-card").forEach(c=>c.classList.remove("cluster-selected"));
  el.classList.add("cluster-selected"); cur=id; loadCluster(id);
}}
function loadCluster(id){{
  const d=clusterData[id];
  document.getElementById("clusterTitle").textContent="Cluster "+id+" - Technology Group";
  document.getElementById("clusterDesc").innerHTML="<strong>"+d.length.toLocaleString()+"</strong> patent families in this t-SNE cluster";
  buildTable(d);
  document.getElementById("clusterData").style.display="block";
  document.getElementById("clusterData").scrollIntoView({{behavior:"smooth"}});
}}
function buildTable(d){{
  if(dt){{dt.destroy();}}
  const tb=document.getElementById("patentTbody"); tb.innerHTML="";
  d.forEach(item=>{{
    const r=tb.insertRow();
    r.insertCell(0).textContent=item.docdb_family_id;
    const pc=r.insertCell(1);
    if(item.publication_number && item.publication_number!=="N/A"){{
      pc.innerHTML="<a href=\\"https://worldwide.espacenet.com/patent/search?q="+item.publication_number+"\\" target=\\"_blank\\" class=\\"pub-link\\">"+item.publication_number+"</a>";
    }} else {{ pc.textContent="N/A"; }}
    const ac=r.insertCell(2);
    let bc="bg-secondary";
    if(item.authority=="EP")bc="bg-primary"; else if(item.authority=="WO")bc="bg-info"; else if(item.authority=="US")bc="bg-success"; else if(item.authority=="CN")bc="bg-warning";
    ac.innerHTML="<span class=\\"badge "+bc+"\\">"+item.authority+"</span>";
    r.insertCell(3).textContent=item.year||"N/A";
    const tc=r.insertCell(4); const t=item.title||"";
    tc.textContent=t.length>100?t.substring(0,100)+"...":t; tc.title=t;
    const apc=r.insertCell(5); const ap=item.applicant||"";
    apc.textContent=ap.length>50?ap.substring(0,50)+"...":ap; apc.title=ap;
  }});
  dt=$("#patentTable").DataTable({{pageLength:25, lengthMenu:[[10,25,50,100,-1],[10,25,50,100,"All"]], order:[[1,"asc"]], columnDefs:[{{targets:[4,5],orderable:false}}]}});
}}
function csvQ(s){{ var q=String(s||"").split(String.fromCharCode(34)).join(String.fromCharCode(34,34)); return String.fromCharCode(34)+q+String.fromCharCode(34); }}
function exportCSV(){{
  if(!cur){{alert("Select a cluster first");return;}}
  const d=clusterData[cur];
  let csv="data:text/csv;charset=utf-8,Family ID,Publication,Authority,Year,Title,Applicant\\n";
  d.forEach(item=>{{ csv+=[item.docdb_family_id,item.publication_number,item.authority,item.year||"N/A",csvQ(item.title),csvQ(item.applicant)].join(",")+"\\n"; }});
  const a=document.createElement("a"); a.setAttribute("href",encodeURI(csv)); a.setAttribute("download","antibiotic_resistance_cluster_"+cur+"_patents.csv");
  document.body.appendChild(a); a.click(); document.body.removeChild(a);
}}
</script></body></html>"""

with open("3_additional_analyses_and_report_output/cluster_explorer.html", "w", encoding="utf-8") as f:
    f.write(widget_html)
print(f"Saved cluster explorer widget, {len(widget_html):,} bytes")

Saved cluster explorer widget, 919,031 bytes


## Step 6: IPC/Technology Co-occurrence Temporal Evolution

Reuses the significant IPC codes + technology labels already established in
`Antibiotic_Resistance_Network_Analysis.ipynb` (`Technology_Legend.xlsx`), splitting
co-occurrence counts into the same 3 periods Analysis 9 already uses (2000-2009,
2010-2016, 2017-2023), to see which technology pairs are growing vs. declining.

In [12]:
from sqlalchemy.orm import aliased
TLS209_APPLN_IPC_2 = aliased(TLS209_APPLN_IPC)

co_occurrence_query = (
    db.query(
        TLS201_APPLN.docdb_family_id.label('patent_family_id'),
        TLS201_APPLN.earliest_filing_year.label('earliest_filing_year'),
        TLS209_APPLN_IPC.ipc_class_symbol.label('IPC_1'),
        TLS209_APPLN_IPC_2.ipc_class_symbol.label('IPC_2')
    )
    .join(TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id)
    .join(TLS209_APPLN_IPC_2, TLS201_APPLN.appln_id == TLS209_APPLN_IPC_2.appln_id)
    .filter(TLS201_APPLN.docdb_family_id.in_(base_family_ids))
    .filter(TLS201_APPLN.earliest_filing_year >= 2000)
    .filter(and_(
        TLS209_APPLN_IPC.ipc_class_symbol > TLS209_APPLN_IPC_2.ipc_class_symbol,
        func.left(TLS209_APPLN_IPC.ipc_class_symbol, 8) != func.left(TLS209_APPLN_IPC_2.ipc_class_symbol, 8)
    ))
)
result = co_occurrence_query.all()
df_cooccur = pd.DataFrame(result, columns=['patent_family_id', 'earliest_filing_year', 'IPC_1', 'IPC_2'])
print(f"Raw co-occurrence records: {len(df_cooccur):,}")
df_cooccur.to_parquet("3_additional_analyses_and_report_output/raw_cooccurrence_with_year.parquet")

Raw co-occurrence records: 204,703


In [13]:
ipc_to_tech_field = {
    'A61K  31': 'Antibiotic Drugs - Organic Compounds', 'A61K  38': 'Antibiotic Drugs - Peptides',
    'A61K  39': 'Antibiotic Drugs - Antigens/Antibodies', 'A61K  45': 'Antibiotic Drug Combinations',
    'A61K  47': 'Antibiotic Drug Formulations', 'A61K   9': 'Antibiotic Drug Delivery',
    'A61P  31': 'Antiinfective Therapeutics', 'A61P  43': 'Drug Screening Methods',
    'C12Q   1': 'Microorganism Detection & Testing', 'C12Q   3': 'Microbial Activity Testing',
    'G01N  33': 'Diagnostic Analysis Methods', 'G01N  27': 'Electrochemical Analysis',
    'G01N  21': 'Optical Analysis', 'G01N  15': 'Particle Analysis',
    'C12N   1': 'Bacteria & Microorganisms', 'C12N   5': 'Cell Culture',
    'C12N  15': 'Genetic Engineering', 'C12N   9': 'Enzymes',
    'C12P   1': 'Antibiotic Fermentation', 'C12P  17': 'Peptide Production', 'C12P  21': 'Protein Production',
    'C07D': 'Antibiotic Chemistry - Heterocyclic', 'C07K': 'Antibiotic Chemistry - Peptides',
    'C07K   7': 'Short Peptide Antibiotics', 'C07K  14': 'Therapeutic Peptides', 'C07K  16': 'Antibody Therapeutics',
    'C07C': 'Antibiotic Chemistry - Organic', 'C07H': 'Antibiotic Chemistry - Carbohydrates',
    'G01N': 'Analytical Testing', 'A61B': 'Medical Diagnostic Devices',
    'A61L': 'Antimicrobial Materials & Sterilization', 'A61L  31': 'Antimicrobial Coatings',
    'A61L   2': 'Sterilization Methods', 'A61M': 'Medical Treatment Devices',
    'C01B': 'Inorganic Compounds', 'C01C': 'Metal Compounds', 'C01G': 'Metal Compounds',
    'C12': 'Biochemistry & Biotechnology', 'A01': 'Agriculture & Biocides', 'A23': 'Food Preservation',
    'A61': 'Medical & Pharmaceutical Technology', 'B01': 'Chemical Processing', 'B82': 'Nanotechnology',
    'C02': 'Water Treatment', 'C07': 'Organic Chemistry', 'C08': 'Polymer Chemistry',
    'G01': 'Measuring & Testing', 'G06': 'Computing & Data Processing',
}

def get_technology_field(ipc_code):
    if not ipc_code: return "Unknown Technology"
    ipc_code = str(ipc_code).strip()
    for n in (8, 4, 3):
        if len(ipc_code) >= n and ipc_code[:n] in ipc_to_tech_field:
            return ipc_to_tech_field[ipc_code[:n]]
    return f"Technology Field ({ipc_code[:3]})"

def get_simplified_tech_name(tech_field, ipc_code):
    simplifications = {
        'Antibiotic Drugs - Organic Compounds': 'Antibiotic Drugs (Organic)', 'Antibiotic Drugs - Peptides': 'Antibiotic Drugs (Peptides)',
        'Antibiotic Drugs - Antigens/Antibodies': 'Antibodies & Antigens', 'Antibiotic Drug Combinations': 'Drug Combinations',
        'Antibiotic Drug Formulations': 'Drug Formulations', 'Antibiotic Drug Delivery': 'Drug Delivery',
        'Antiinfective Therapeutics': 'Antiinfective Therapy', 'Microorganism Detection & Testing': 'Microbial Testing',
        'Microbial Activity Testing': 'Activity Testing', 'Diagnostic Analysis Methods': 'Diagnostic Analysis',
        'Bacteria & Microorganisms': 'Bacteria', 'Genetic Engineering': 'Genetic Engineering',
        'Antibiotic Chemistry - Heterocyclic': 'Chemistry (Heterocyclic)', 'Antibiotic Chemistry - Peptides': 'Chemistry (Peptides)',
        'Antibiotic Chemistry - Organic': 'Chemistry (Organic)', 'Antimicrobial Materials & Sterilization': 'Antimicrobial Materials',
        'Medical & Pharmaceutical Technology': 'Medical Technology', 'Biochemistry & Biotechnology': 'Biochemistry',
    }
    return simplifications.get(tech_field, tech_field)

legend = pd.read_excel("2_technology_network_output/technology_legend.xlsx")
significant_ipc_list = list(set(c.strip() for codes in legend['IPC_Code'] for c in str(codes).split(',')))
print(f"Significant IPC codes reused from existing legend: {len(significant_ipc_list)}")

Significant IPC codes reused from existing legend: 61


In [14]:
df_cooccur['IPC_1_8'] = df_cooccur['IPC_1'].astype(str).str[:8]
df_cooccur['IPC_2_8'] = df_cooccur['IPC_2'].astype(str).str[:8]
df_f = df_cooccur[df_cooccur['IPC_1_8'].isin(significant_ipc_list) & df_cooccur['IPC_2_8'].isin(significant_ipc_list)].copy()
df_f['pair'] = df_f.apply(lambda r: tuple(sorted([r['IPC_1_8'], r['IPC_2_8']])), axis=1)
df_f = df_f.drop_duplicates(subset=['patent_family_id', 'pair'])

def simplified_pair(row):
    a, b = row['pair']
    ta = get_simplified_tech_name(get_technology_field(a), a)
    tb = get_simplified_tech_name(get_technology_field(b), b)
    return tuple(sorted([ta, tb]))

df_f['tech_pair'] = df_f.apply(simplified_pair, axis=1)

eras = {"2000-2009": (2000, 2009), "2010-2016": (2010, 2016), "2017-2023": (2017, 2023)}
era_counts = {era: df_f[(df_f['earliest_filing_year'] >= y0) & (df_f['earliest_filing_year'] <= y1)].groupby('tech_pair').size() for era, (y0, y1) in eras.items()}

cooccur = pd.read_excel("2_technology_network_output/ipc_cooccurrence.xlsx")
cooccur['tech_pair'] = cooccur.apply(lambda r: tuple(sorted([get_simplified_tech_name(r['Tech_Field_A'], r['IPC_A']), get_simplified_tech_name(r['Tech_Field_B'], r['IPC_B'])])), axis=1)
top20 = cooccur.groupby('tech_pair')['cooccurrence_count'].sum().sort_values(ascending=False).head(20)

rows = []
for pair, whole_count in top20.items():
    row = {"Pair": f"{pair[0]} <-> {pair[1]}", "Whole-period": int(whole_count)}
    for era_name in eras:
        row[era_name] = int(era_counts[era_name].get(pair, 0))
    rows.append(row)
result_df = pd.DataFrame(rows)
result_df["Growth"] = result_df["2017-2023"] - result_df["2000-2009"]
result_df["Direction"] = result_df["Growth"].apply(lambda g: "GROWING" if g > 0 else ("DECLINING" if g < 0 else "STABLE"))
result_df = result_df.sort_values("Growth", ascending=False)
result_df.to_excel("3_additional_analyses_and_report_output/temporal_pair_evolution.xlsx", index=False)
print(result_df.to_string(index=False))

                                                      Pair  Whole-period  2000-2009  2010-2016  2017-2023  Growth Direction
              Antiinfective Therapy <-> Medical Technology          2880        139        260        841     702   GROWING
                 Medical Technology <-> Medical Technology          2837        202        289        805     603   GROWING
      Antibiotic Drugs (Organic) <-> Antiinfective Therapy          4524        195        291        592     397   GROWING
                  Food Preservation <-> Medical Technology           830         19        158        349     330   GROWING
         Antibiotic Drugs (Organic) <-> Medical Technology          3050        107        194        394     287   GROWING
                      Drug Delivery <-> Medical Technology           879         50         83        240     190   GROWING
                  Drug Formulations <-> Medical Technology          1103         43         72        205     162   GROWING
        

In [15]:
eras_list = list(eras.keys())
z = result_df[eras_list].values
fig = go.Figure(data=go.Heatmap(
    z=z, x=eras_list, y=result_df["Pair"], colorscale="Plasma",
    text=z, texttemplate="%{text}", textfont={"size": 11}, colorbar=dict(title="Co-occurrences"),
))
fig.update_layout(
    title="Antibiotic Resistance -- Technology Pair Co-occurrence Evolution (Top 20 Pairs)",
    xaxis_title="Period", yaxis_title="Technology Pair", width=1100, height=750, margin=dict(l=350),
)
fig.write_html("3_additional_analyses_and_report_output/relationship_change_heatmap.html")
n_growing = (result_df["Direction"] == "GROWING").sum()
n_declining = (result_df["Direction"] == "DECLINING").sum()
print(f"GROWING pairs ({n_growing} of {len(result_df)}), DECLINING pairs ({n_declining} of {len(result_df)})")

GROWING pairs (18 of 20), DECLINING pairs (2 of 20)


## Step 7 &middot; Assemble the one-file report

Each step above wrote a standalone HTML file, and every Plotly one carries its own
4.6&nbsp;MB copy of the plotly.js library &mdash; thirteen files, 37&nbsp;MB, no way to move
between them.

This step stitches them into **one report you page through** with Previous / Next.
The four Plotly figures are still live objects at this point, so they go in as bare
`<div>`s sharing a single embedded copy of the library. The two hand-built widgets are
complete documents with their own dependencies, so each keeps its own document in an
iframe. The seven analyses that came from the original pipeline are not rebuilt here
&mdash; they are referenced from the folder next to the report, so they stay current.

In [16]:
# ── Step 7 · Assemble the one-file report ──────────────────────────────────────
#
# Every step above wrote its own HTML file. This cell stitches them into a single
# report you page through with Previous / Next.
#
# Three kinds of page, handled differently on purpose:
#   * the four Plotly figures are still live objects right here, so they go in as
#     bare <div>s sharing ONE embedded copy of plotly.js. Each standalone file
#     carries its own 4.6 MB copy — that duplication is most of the folder's weight.
#   * the two hand-built widgets are complete HTML documents with their own CDN
#     dependencies, so each keeps its own document inside an iframe.
#   * the seven analyses from the original pipeline are not rebuilt here. They are
#     referenced from the folder next to this file, so they stay current if that
#     pipeline produces new ones.

import json
import os
from pathlib import Path
from plotly.offline import get_plotlyjs

BUNDLE_PATH  = Path("report/antibiotic_resistance_report.html")
REPORT_DIR   = BUNDLE_PATH.parent   # the report lives in report/, so every
def rel(p):                          # on-disk path it links must be made
    return os.path.relpath(p, REPORT_DIR)   # relative to that folder, not to cwd
FINAL_REPORT = "0_inputs/Antibiotic_Report_FINAL.html"
CLEAN   = "1_dataset_and_search_strategy_output"
TSNE    = "0_inputs"
NETWORK = "2_technology_network_output"

# kind, title, source — in reading order, not in the order they were computed
PAGES = [

    ("plotly", "Triadic families &mdash; world map",   triadic_map_fig),
    ("doc",    "Triadic families &mdash; ranking",     table_html),
    ("plotly", "Filing authorities &mdash; totals",    fig1),
    ("plotly", "Filing authorities &mdash; trend",     fig2),
    ("file",   "Technology clusters &mdash; scatter",  "0_inputs/interactive_scatter.html"),
    ("file",   "Technology clusters &mdash; dashboard", "0_inputs/cluster_dashboard.html"),
    ("doc",    "Cluster explorer",                     widget_html),
    ("plotly", "IPC co-occurrence over time",          fig),
    ("file",   "Technology network",                   "2_technology_network_output/technology_network.html"),
    ("file",   "IPC analysis",                         "1_dataset_and_search_strategy_output/ipc_analysis.html"),
    ("file",   "Database statistics",                  "1_dataset_and_search_strategy_output/statistics.html"),
    ("file",   "Full dataset (3,974 families)",        "1_dataset_and_search_strategy_output/dataset_highlighted.html"),
]

# ── turn each page into markup ────────────────────────────────────────────────
panels, docs, meta = [], {}, []

for i, (kind, title, source) in enumerate(PAGES):
    meta.append({"title": title, "kind": kind})
    if kind == "plotly":
        # include_plotlyjs=False: the library is embedded once, in <head>
        body = source.to_html(include_plotlyjs=False, full_html=False,
                              div_id=f"plot{i}", default_height="100%")
    elif kind == "doc":
        docs[str(i)] = source          # complete document -> iframe srcdoc
        body = f'<iframe class="page-frame" data-doc="{i}"></iframe>'
    else:
        href = rel(source)
        body = (f'<iframe class="page-frame" data-src="{href}"></iframe>'
                f'<div class="fallback">Frame empty? '
                f'<a href="{href}" target="_blank">open this page on its own &rarr;</a></div>')
    panels.append(f'<section class="page" id="page{i}" hidden>{body}</section>')

steps = "".join(
    f'<button class="step" data-go="{i}" title="{p["title"]}">{i + 1}</button>'
    for i, p in enumerate(meta))

# JSON inside <script> must not contain a literal </script
blob = json.dumps({"meta": meta, "docs": docs}).replace("</script", r"<\/script")

TEMPLATE = """<!doctype html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Antibiotic Resistance &mdash; Patent Landscape</title>
<style>
  :root { --accent:#be0f05; --ink:#334155; --muted:#94a3b8; --line:#e2e8f0; --bg:#fff; --panel:#f8fafc; }
  @media (prefers-color-scheme: dark) {
    :root { --ink:#e2e8f0; --muted:#94a3b8; --line:#334155; --bg:#0f172a; --panel:#1e293b; }
  }
  * { box-sizing:border-box; }
  body { margin:0; font-family:system-ui,-apple-system,sans-serif; color:var(--ink);
         background:var(--bg); display:flex; flex-direction:column; height:100vh; }
  header { padding:14px 24px; border-bottom:1px solid var(--line); flex:none; }
  .brand { display:inline-block; background:var(--accent); color:#fff; padding:6px 12px;
           border-radius:8px; font-weight:800; letter-spacing:-.3px; }
  .sub { font-size:12px; color:var(--muted); margin-top:6px; }
  .steps { display:flex; flex-wrap:wrap; gap:4px; padding:10px 24px; border-bottom:1px solid var(--line);
           background:var(--panel); flex:none; overflow-x:auto; }
  .step { border:1px solid var(--line); background:var(--bg); color:var(--muted); cursor:pointer;
          width:30px; height:30px; border-radius:6px; font-size:12px; font-weight:600; flex:none; }
  .step:hover { border-color:var(--accent); color:var(--accent); }
  .step.on { background:var(--accent); border-color:var(--accent); color:#fff; }
  main { flex:1 1 auto; overflow:auto; position:relative; }
  .page { height:100%; display:flex; flex-direction:column; }
  .page[hidden] { display:none; }
  .page-frame { flex:1 1 auto; width:100%; border:0; display:block; }
  .fallback { flex:none; padding:6px 24px; font-size:12px; color:var(--muted);
              border-top:1px solid var(--line); }
  .fallback a { color:var(--accent); font-weight:600; }
  footer { display:flex; align-items:center; gap:16px; padding:12px 24px;
           border-top:1px solid var(--line); background:var(--panel); flex:none; }
  button.nav { background:var(--accent); color:#fff; border:0; border-radius:8px;
               padding:9px 18px; font-weight:600; cursor:pointer; font-size:14px; }
  button.nav[disabled] { opacity:.35; cursor:default; }
  .where { font-size:13px; color:var(--muted); flex:1; }
  .where b { color:var(--ink); }
  a.final { font-size:12px; color:var(--accent); text-decoration:none; font-weight:600; }
</style>
<script>@@PLOTLYJS@@</script>
<script type="application/json" id="bundle-data">@@BLOB@@</script>
</head>
<body>
<header>
  <div class="brand">TIP4PATLIBS &ndash; Antibiotic Resistance</div>
  <div class="sub">Patent landscape &middot; created by Riccardo Priore, Centro PATLIB, AREA Science Park
    &middot; <a class="final" href="@@FINAL@@" target="_blank">open the full original report &rarr;</a></div>
</header>
<div class="steps">@@STEPS@@</div>
<main>@@PANELS@@</main>
<footer>
  <button class="nav" id="prev">&larr; Previous</button>
  <button class="nav" id="next">Next &rarr;</button>
  <div class="where"><b id="pos"></b> &nbsp;<span id="label"></span></div>
</footer>
<script>
(function () {
  var data  = JSON.parse(document.getElementById('bundle-data').textContent);
  var meta  = data.meta, docs = data.docs, cur = -1;
  var pages = document.querySelectorAll('.page');
  var steps = document.querySelectorAll('.step');

  function show(i) {
    if (i < 0 || i >= meta.length || i === cur) return;
    if (cur >= 0) { pages[cur].hidden = true; steps[cur].classList.remove('on'); }
    cur = i;
    var page = pages[i];
    page.hidden = false;
    steps[i].classList.add('on');

    // fill iframes on first view — keeps the initial load fast
    var frame = page.querySelector('.page-frame');
    if (frame && !frame.dataset.loaded) {
      frame.dataset.loaded = '1';
      if (frame.dataset.doc !== undefined) frame.srcdoc = docs[frame.dataset.doc];
      else frame.src = frame.dataset.src;
    }
    // a Plotly figure laid out while hidden has no size yet
    var plot = page.querySelector('.plotly-graph-div');
    if (plot && window.Plotly) Plotly.Plots.resize(plot);

    document.getElementById('pos').textContent   = (i + 1) + ' / ' + meta.length;
    document.getElementById('label').innerHTML   = meta[i].title;
    document.getElementById('prev').disabled     = (i === 0);
    document.getElementById('next').disabled     = (i === meta.length - 1);
    if (location.hash !== '#' + (i + 1)) history.replaceState(null, '', '#' + (i + 1));
  }

  document.getElementById('prev').onclick = function () { show(cur - 1); };
  document.getElementById('next').onclick = function () { show(cur + 1); };
  steps.forEach(function (b) { b.onclick = function () { show(+b.dataset.go); }; });
  document.addEventListener('keydown', function (e) {
    if (e.target.tagName === 'INPUT' || e.target.tagName === 'TEXTAREA') return;
    if (e.key === 'ArrowLeft')  show(cur - 1);
    if (e.key === 'ArrowRight') show(cur + 1);
  });
  window.addEventListener('resize', function () {
    var plot = pages[cur] && pages[cur].querySelector('.plotly-graph-div');
    if (plot && window.Plotly) Plotly.Plots.resize(plot);
  });

  var start = parseInt((location.hash || '#1').slice(1), 10);
  show(isNaN(start) || start < 1 || start > meta.length ? 0 : start - 1);
})();
</script>
</body></html>
"""

html_out = (TEMPLATE
            .replace("@@PLOTLYJS@@", get_plotlyjs())
            .replace("@@BLOB@@", blob)
            .replace("@@STEPS@@", steps)
            .replace("@@PANELS@@", "".join(panels))
            .replace("@@FINAL@@", rel(FINAL_REPORT)))

BUNDLE_PATH.write_text(html_out, encoding="utf-8")

built = sum(1 for k, _, _ in PAGES if k != "file")
print(f"{BUNDLE_PATH}  —  {BUNDLE_PATH.stat().st_size / 1e6:.1f} MB, {len(PAGES)} pages "
      f"({built} built here, {len(PAGES) - built} referenced from the folder)")

# ── Open it ───────────────────────────────────────────────────────────────────
# JupyterLab's own HTML viewer renders a file inside a sandboxed frame: the shell
# appears and paging works, but every embedded page stays blank. The launcher serves
# the report through jupyter-server-proxy instead, which has no such restriction.
import sys

repo = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "CLAUDE.md").exists()), Path.cwd())
sys.path.insert(0, str(repo / "1_startwithtip"))
from tip_tools import open_html

open_html(BUNDLE_PATH, "the paged report")


report/antibiotic_resistance_report.html  —  5.8 MB, 12 pages (6 built here, 6 referenced from the folder)


## Done

All outputs land in this notebook's `3_additional_analyses_and_report_output/` folder,
alongside the other notebooks' results, and Step&nbsp;7 assembles them into
`Antibiotic_Resistance_Report.html` &mdash; the paged report to open after a run.

The original `Antibiotic_Report_FINAL.html` stays untouched and is linked from the
report's header; it remains the canonical published version.